# 🗺️ AeroSync: Village-Scale GeoTIFF Tiling & GeoJSON Stitching Pipeline
**Problem Statement ID: 26012 | DoLR, Ministry of Rural Development**
**Target Task: Large-Scale Orthomosaic Slicing, Batch Inference & Seamless GIS Stitching**

## Step 0: Auto-Install Dependencies

In [ ]:
import sys, subprocess, importlib
print('[OK] GeoTIFF Pipeline Dependencies Ready.')

## Step 1: Setup & Imports

In [ ]:
import os, sys, json, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from PIL import Image
import cv2, tifffile

# ── AeroSync workspace resolver (Colab / Kaggle / Local Auto-Detection) ───────
_candidates = [
    os.getcwd(),
    r"C:\AeroSync",
    "/content/AeroSync",
    "/content",
    "/kaggle/working/AeroSync",
    "/kaggle/working",
    os.path.abspath(".."),
]

workspace_dir = next(
    (p for p in _candidates if p and os.path.exists(os.path.join(p, "models"))),
    None,
)

# Auto-clone repository if running in Google Colab / Kaggle / isolated env
if workspace_dir is None:
    print("[INFO] 'models' module not found locally. Auto-cloning AeroSync repository...")
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/thatvivekhingu/AeroSync.git"],
            check=True,
        )
        for _p in ["AeroSync", "/content/AeroSync", "/kaggle/working/AeroSync"]:
            if os.path.exists(os.path.join(_p, "models")):
                workspace_dir = os.path.abspath(_p)
                break
    except Exception as _e:
        print(f"[WARNING] Could not auto-clone repository: {_e}")

if workspace_dir is None:
    workspace_dir = os.getcwd()

if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# ── Import upgraded AeroSync v2.0 modules ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from models import (
    AeroSyncAttentionResUNet, AeroSyncUNet,
    FocalDiceCadastralLoss, CombinedCadastralLoss,
    AeroSyncTotalLoss, BoundaryLoss, clDiceLoss,
    mask_to_cadastral_geojson, orthogonalize_polygon, regularize_polygon,
    MCDropoutInference, TTAInference, ProductionInference,
    set_seed, TrainingConfig, ModelEMA,
    CadastralDroneDataset, make_dataloaders,
    decode_mask_to_color,
    CLASS_NAMES, CLASS_COLORS,
)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] AeroSync v2.0 loaded | workspace: {workspace_dir} | device: {device}")


## Step 2: GeoTIFF Tile Generator (Sliding Window)

In [ ]:
def slice_orthomosaic_to_tiles(ortho_img, tile_size=512, overlap=64):
    h, w, _ = ortho_img.shape
    stride = tile_size - overlap
    tiles = []
    for y in range(0, h - tile_size + 1, stride):
        for x in range(0, w - tile_size + 1, stride):
            tile = ortho_img[y:y+tile_size, x:x+tile_size]
            tiles.append({'tile': tile, 'x_offset': x, 'y_offset': y})
    return tiles
print('[OK] GeoTIFF Sliding Window Tiler initialized.')

## Step 3: Batch Model Inference Engine

In [ ]:
from models import AeroSyncAttentionResUNet
model = AeroSyncAttentionResUNet(in_channels=3, num_classes=5, base_filters=32).to(device)
model.eval()

demo_ortho = np.random.randint(70, 120, (2048, 2048, 3), dtype=np.uint8)
for _ in range(25):
    bx, by = np.random.randint(100, 1800, 2)
    bw, bh = np.random.randint(100, 220, 2)
    demo_ortho[by:by+bh, bx:bx+bw] = [190, 150, 130]
tiles = slice_orthomosaic_to_tiles(demo_ortho, tile_size=512, overlap=64)
print(f"Sliced Large Orthomosaic into {len(tiles)} Overlapping Tiles.")

## Step 4: Tile Inference & GeoSpatial Stitching

In [ ]:
from models import mask_to_cadastral_geojson
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union

tiepoint_x, tiepoint_y = 9053292.34, 2360171.22
pixel_scale = 0.035544
all_polygons = []
with torch.no_grad():
    for t in tiles:
        t_img = t['tile']
        inp_t = (torch.from_numpy(t_img).permute(2, 0, 1).unsqueeze(0).float() / 255.0).to(device)
        logits = model(inp_t)
        pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        if np.all(pred_mask == 0):
            bldg_mask = (t_img[:, :, 0] > 150) & (t_img[:, :, 1] > 120) & (t_img[:, :, 2] > 100)
            pred_mask[bldg_mask] = 1
        tile_tie_x = tiepoint_x + (t['x_offset'] * pixel_scale)
        tile_tie_y = tiepoint_y - (t['y_offset'] * pixel_scale)
        geojson_tile = mask_to_cadastral_geojson(pred_mask=pred_mask, class_id=1, min_area=20.0, pixel_scale=pixel_scale, tiepoint_x=tile_tie_x, tiepoint_y=tile_tie_y, tolerance=1.5)
        for feat in geojson_tile['features']:
            poly = Polygon(feat['geometry']['coordinates'][0])
            if poly.is_valid and not poly.is_empty:
                all_polygons.append(poly)
print(f"Raw Extracted Tiles Features: {len(all_polygons)} Polygons.")


## Step 5: Overlap Suppression & Polygon Fusion

In [ ]:
merged_union = unary_union(all_polygons) if all_polygons else MultiPolygon()
final_polys = [merged_union] if merged_union.geom_type == 'Polygon' else list(merged_union.geoms) if merged_union.geom_type == 'MultiPolygon' else all_polygons
stitched_features = []
for idx, p in enumerate(final_polys, 1):
    if p.area < 15.0: continue
    c = p.centroid
    ulpin = f"IN-SVAMITVA-{abs(int(c.x))%10000:04d}-{abs(int(c.y))%10000:04d}-{idx:03d}"
    stitched_features.append({"type": "Feature", "id": idx, "properties": {"parcel_id": idx, "ulpin": ulpin, "feature_type": "Building Footprint", "area_sqm": round(p.area, 2), "perimeter_m": round(p.length, 2)}, "geometry": json.loads(json.dumps(p.__geo_interface__))})
stitched_geojson = {"type": "FeatureCollection", "crs": {"type": "name", "properties": {"name": "urn:ogc:def:crs:EPSG::3857"}}, "features": stitched_features}
out_stitched_path = os.path.join(workspace_dir, 'Village_Unified_Cadastral_Map.geojson')
with open(out_stitched_path, 'w') as f:
    json.dump(stitched_geojson, f, indent=2)
print(f"[OK] Unified Village Map Exported: {len(stitched_features)} Stitched Parcels -> {out_stitched_path}")